<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/InceptionV3%2BResUNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:
# Load the data
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

In [7]:
import numpy as np
import tensorflow as tf

# 1. Define the ESA class map (6 merged classes)
class_map = {
    10: 0, # Tree cover
    20: 1, # Shrubland
    30: 2, # Grassland
    40: 3, # Cropland
    60: 3, # Bare / Sparse vegetation (Combined with Cropland)
    50: 4, # Built-up
    80: 5, # Permanent water bodies
    90: 5, # Herbaceous wetland (Combined with Water Bodies)
}

# 2. Use 101 to avoid IndexError if metadata values exist
lut = np.full(101, -1, dtype=np.int32)
for old_id, new_id in class_map.items():
    lut[old_id] = new_id

# 3. Apply mapping
Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]

# Ensure the (..., 1) dimension is preserved for Sparse Loss
if Y_train_ready.ndim == 3:
    Y_train_ready = np.expand_dims(Y_train_ready, axis=-1)
    Y_test_ready = np.expand_dims(Y_test_ready, axis=-1)

print(f"Unique IDs in merged train mask: {np.unique(Y_train_ready)}")
print(f"Final Y_train_ready shape: {Y_train_ready.shape}")

Unique IDs in merged train mask: [0 1 2 3 4 5]
Final Y_train_ready shape: (639, 256, 256, 1)


In [8]:
# Converting for RAM efficiency

# Imagery must be float32 for the model weights
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

# Labels must be int32 for Sparse Categorical Crossentropy
Y_train_ready = Y_train_ready.astype('int32')
Y_test_ready = Y_test_ready.astype('int32')

In [12]:
def residual_block(x, filters, dropout_rate=0.3):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dropout(dropout_rate)(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_inception_res_unet(input_shape=(256, 256, 7), num_classes=6):
    inputs = layers.Input(input_shape)

    # 1. THE 7-BAND ADAPTER
    adapter = layers.Conv2D(3, (1, 1), padding='same', name='band_adapter')(inputs)

    # 2. INCEPTIONV3 ENCODER
    # Clear session to try and reset layer names, but we'll use robust indexing just in case
    inception_base = tf.keras.applications.InceptionV3(
        include_top=False,
        weights='imagenet',
        input_shape=(256, 256, 3)
    )

    # Using specific layer names from your error log to be safe:
    # 'activation_94' is the first 125x125 skip
    # 'activation_96' is a 61x61 skip
    # 'mixed0' or 'mixed2' are standard 28x28 skips

    s1 = inception_base.get_layer(index=11).output    # ~125x125 (Initial Conv/Act)
    s2 = inception_base.get_layer(index=17).output    # ~61x61
    s3 = inception_base.get_layer("mixed2").output    # 28x28
    bridge = inception_base.get_layer("mixed7").output # 12x12

    encoder_model = models.Model(inputs=inception_base.input, outputs=[s1, s2, s3, bridge])
    enc_s1, enc_s2, enc_s3, enc_bridge = encoder_model(adapter)

    # 3. RESIDUAL DECODER
    u1 = layers.UpSampling2D((2, 2))(enc_bridge)
    u1 = layers.Resizing(enc_s3.shape[1], enc_s3.shape[2])(u1)
    u1 = layers.concatenate([u1, enc_s3])
    d1 = residual_block(u1, 128)

    u2 = layers.UpSampling2D((2, 2))(d1)
    u2 = layers.Resizing(enc_s2.shape[1], enc_s2.shape[2])(u2)
    u2 = layers.concatenate([u2, enc_s2])
    d2 = residual_block(u2, 64)

    u3 = layers.UpSampling2D((2, 2))(d2)
    u3 = layers.Resizing(enc_s1.shape[1], enc_s1.shape[2])(u3)
    u3 = layers.concatenate([u3, enc_s1])
    d3 = residual_block(u3, 32)

    u_final = layers.UpSampling2D((2, 2))(d3)
    u_final = layers.Resizing(256, 256)(u_final)

    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(u_final)

    return models.Model(inputs, outputs, name="InceptionV3_ResUNet")

# Clear the Keras backend to reset '_94' suffixes
tf.keras.backend.clear_session()

model = build_inception_res_unet()
print("Model built successfully!")

Model built successfully!


In [14]:
import numpy as np
from sklearn.utils import class_weight

# 1. Calculate Class Weights to handle imbalance (Trees vs Buildings etc.)
# We flatten the Y_train to get a list of all pixel labels
y_flat = Y_train_ready.flatten()
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_flat),
    y=y_flat
)
class_weights_dict = dict(enumerate(weights))
print("Computed Class Weights", weights)

Computed Class Weights [1.11912802 1.041065   0.75546067 0.97782913 1.32307165 0.95812405]


In [ ]:
from sklearn.metrics import f1_score

class TableLogger(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        # val_data is now (X_test, Y_test_ready) where Y is (N, 256, 256, 1)
        self.X_val, self.y_val_integers = val_data

    def on_train_begin(self, logs=None):
        print(f"\n{'Epoch':<6} | {'Train Loss':<10} | {'Val Loss':<10} | {'Val Acc':<8} | {'F1 (Macro)':<10}")
        print("-" * 65)

    def on_epoch_end(self, epoch, logs=None):
        # Predict on validation set
        val_logits = self.model.predict(self.X_val, verbose=0, batch_size=8) # Lower batch size for safety

        # Model output is (N, 256, 256, 6) -> Get the class with highest probability
        val_preds = np.argmax(val_logits, axis=-1).flatten()

        # Ground truth is already (N, 256, 256, 1) -> Just flatten it
        val_true = self.y_val_integers.flatten()

        # Calculate F1 Score (Macro)
        val_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)

        # Get values from logs
        train_loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        val_acc = logs.get('val_accuracy', 0)

        print(f"{epoch+1:<6} | {train_loss:<10.4f} | {val_loss:<10.4f} | {val_acc:<8.4f} | {val_f1:<10.4f}")


In [15]:

# 3. Compile the Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7
)

# Initialize our custom logger with the Sparse (Integer) labels
table_logger = TableLogger(val_data=(X_test, Y_test_ready))

# Start training using Sparse labels and Class Weights
history = model.fit(
    X_train, Y_train_ready,
    validation_data=(X_test, Y_test_ready),
    epochs=25,
    batch_size=8,
    class_weight=class_weights_dict,
    callbacks=[table_logger, reduce_lr],
    verbose=0
)


Model compiled with Sparse Categorical Crossentropy.


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
import gc

print("Generating predictions...")

# 1. Predict and immediately convert to indices to save RAM
# We use a small batch size to stay stable
train_preds = np.argmax(model.predict(X_train, batch_size=8, verbose=1), axis=-1).flatten()
test_preds = np.argmax(model.predict(X_test, batch_size=8, verbose=1), axis=-1).flatten()

# 2. Flatten the ground truth labels
y_train_flat = Y_train_ready.flatten()
y_test_flat = Y_test_ready.flatten()

# 3. Trigger Garbage Collection to clear the heavy logit arrays from memory
gc.collect()

# 4. Calculate Overall Metrics
train_acc = accuracy_score(y_train_flat, train_preds)
test_acc = accuracy_score(y_test_flat, test_preds)
test_f1_macro = f1_score(y_test_flat, test_preds, average="macro")

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1_macro:.4f}")

# 5. Detailed Classification Report
merged_names = [
    "Tree cover",
    "Shrubland",
    "Grassland",
    "Cropland/vegetation",
    "Built-up",
    "Permanent water bodies"
]

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test_flat, test_preds, target_names=merged_names))

In [16]:
# Patch conversion
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

def pixels_to_patch_label(y_array):
    """
    Converts (N, 256, 256, 1) or (N, 256, 256) pixel masks
    into (N,) patch labels using the majority class.
    """
    patch_labels = []
    for i in range(y_array.shape[0]):
        # Flatten the 256x256 patch into a 1D list of pixels
        pixels = y_array[i].flatten()

        # Find the most frequent class (the mode)
        counts = np.bincount(pixels, minlength=6)
        majority_class = np.argmax(counts)
        patch_labels.append(majority_class)

    return np.array(patch_labels)

# 1. Convert Ground Truth (The actual labels)
y_test_patch_true = pixels_to_patch_label(Y_test_ready)

# 2. Convert Your Model's Predictions
# Note: We reshape test_preds back to (N, 256, 256) first
test_preds_spatial = test_preds.reshape(-1, 256, 256)
y_test_patch_pred = pixels_to_patch_label(test_preds_spatial)

# 3. Calculate Patch-Level Accuracy
patch_acc = accuracy_score(y_test_patch_true, y_test_patch_pred)

print(f"Pixel-Level Accuracy: {test_acc:.4f}")
print(f"Patch-Level Accuracy: {patch_acc:.4f}")

print("\n--- PATCH-LEVEL CLASSIFICATION REPORT ---")
print(classification_report(y_test_patch_true, y_test_patch_pred, target_names=merged_names))

NameError: name 'test_preds' is not defined